# 分类梯度与配置编码器：固定配对训练

原特征、原标签、原损失权重、原参数数量；只改变分类头输入的反向传播路径。三seed等权，不选择最优seed。预先固定0/2/5bp完整账户，复用逐值验证的强基线。此假设未预先证明有利。

In [1]:
from pathlib import Path
import os,json
import qlib
from src.workflow.paired_allocator_experiment import run
root=Path.cwd()
while not (root/'SPEC.md').exists():root=root.parent
os.chdir(root);os.environ['MLFLOW_ALLOW_FILE_STORE']='true';card=root/'research/allocation_encoder_routing_20261014';plan=json.loads((card/'plan.json').read_text());qlib.init(provider_uri=str(root/'data/qlib/full_path_tiered_allocation_20261013'),region='us',kernels=1)
summary=run(root=root,card=card,plan=plan)
print({k:summary[k] for k in ['decision','new_fits','new_native_cases','gates']})


# 分类直接梯度停止回传：配对训练与完整回测失败

本轮实际完成3个固定seed候选训练，每个3014个成熟主样本、20741个辅助样本、300轮；固定最终轮、等权平均，不挑最好seed。训练信号从2011-01-03到2022-12-21，最后成熟标签2022-12-30 09:31:00-05:00。复用逐值验证的3个匹配控制模型及9套既有原生账户，新跑0/2/5bp三套候选完整账户。

唯一干预：主分类和辅助分类调用的tail_head/alpha_head对输入detach，分类头自身仍正常学习原标签，共享编码器的直接梯度来自配置损失。恢复原始输入，没有叠加上一轮失败的动量分段。标签定义、分类损失权重、模型参数量、初始参数与前向数值、数据/切分、300轮训练、AdamW、共同梯度裁剪和交易控制器保持原设置。共同梯度裁剪仍可能产生间接耦合，因此不声称两个训练任务已完全独立。

本假设来自此前共享梯度量级不同的诊断，但该诊断近乎正交、未证明负迁移；本轮不预设停止回传有效，也不把失败解释成需要继续找更合适系数。

10万美元、固定IBKR Pro Tiered研究费用投影、0/2/5bp独立持仓及月度费用状态，944个可执行区间（2023-01-03信号起，2026-10-09最终结算）。原模拟盘对照是同起点同费用口径的历史重建，不是实盘已实现收益。

|模型|滑点|累计净收益|CAGR|收盘最大回撤|盘中监控最大回撤|相对SOXX财富变化|
|---|---|---|---|---|---|---|
|分类停止直接回传|0bp|+72.80%|15.72%|-28.71%|-29.22%|-64.57%|
|原共享编码器匹配基线|0bp|+280.24%|42.84%|-25.93%|-26.20%|-22.04%|
|原模拟盘模型重建|0bp|+228.99%|37.42%|-28.11%|-29.50%|-32.55%|
|SOXX|0bp|+387.75%|52.66%|-42.87%|-43.38%|+0.00%|
|分类停止直接回传|2bp|+185.13%|32.27%|-28.26%|-28.78%|-41.54%|
|原共享编码器匹配基线|2bp|+269.16%|41.72%|-24.40%|-25.77%|-24.31%|
|原模拟盘模型重建|2bp|+279.67%|42.78%|-24.17%|-27.07%|-22.16%|
|SOXX|2bp|+387.73%|52.65%|-42.87%|-43.39%|+0.00%|
|分类停止直接回传|5bp|+78.07%|16.65%|-29.09%|-29.60%|-63.49%|
|原共享编码器匹配基线|5bp|+266.08%|41.40%|-24.46%|-25.83%|-24.94%|
|原模拟盘模型重建|5bp|+41.12%|9.63%|-31.38%|-31.42%|-71.06%|
|SOXX|5bp|+387.70%|52.65%|-42.88%|-43.39%|+0.00%|

候选三个成本情景均落后匹配基线与SOXX；2bp +185.13%不能掩盖0bp +72.80%和5bp +78.07%的分化。成本情景各自沿持仓、费用与风控状态演化，不能把2bp较高收益解释成增加交易成本有利；本轮没有通过额外回放定位某一个阈值为唯一原因。候选2/5bp实际平均现金46.72%/52.94%，也没有形成可靠的低现金、高净超额改进。

|信号年度|2bp候选净收益|2bp相对SOXX财富变化|5bp候选净收益|5bp相对SOXX财富变化|
|---|---|---|---|---|
|2023|+41.54%|-9.62%|+40.82%|-10.06%|
|2024|+19.34%|-1.35%|+18.80%|-1.80%|
|2025|+9.05%|-25.71%|-12.65%|-40.50%|
|2026|+54.79%|-11.74%|+21.85%|-30.53%|

相对财富变化是增长倍数之比减1，不是累计收益百分点。收盘/盘中30%回撤和相对SOXX的CVaR门通过，但净超额、匹配基线、原模型全成本比较、年度/126日滚动稳定性与区块置信门均失败。历史已被反复查看，独立开发和前向资格也未取得。

决定：not_promoted_fixed_encoder_routing_failed。关闭这一固定干预，不继续扫分类系数、detach位置、seed、epoch或特征来挽救。结果不支持“分类梯度较大，所以应阻止它更新配置表示”的优化方案，也不证明所有辅助学习必然有益。现役保留原方案，稳定净超额目标未完成。

下一项训练必须有不同且未检验的机制依据，仍用强配对基线和完整净费账户；不能把本轮2bp的相对较好路径当成选择成本或控制参数的依据。继续历史训练回测，不恢复实时采集等待。

验证：一个集中合同测试覆盖前向数值及参数量不变、主/辅助分类对编码器直接梯度为零、分类头可学习、配置梯度保留；1 passed。实际运行核对初始张量哈希、原始scaler、成熟训练域和基线预测逐值一致。公共配对执行流程复用原算法，上一轮入口兼容导入检查通过，无需重训旧实验。NAS冷读回见readback_verification.json。

[完整净费比较](backtest_benchmark_comparison.csv) · [年度结果](annual_metrics.csv) · [区块区间](uncertainty.csv) · [固定计划](plan.json)

Recorder `92b2850803bd4cba9bd9cc1f7c3abefe`。
